# DATA-511: Homework Assignment 8

## Instructions

- This homework assignment is worth **42** points. 
- Please strive for clarity and organization.
- **AI Usage**:
   - AI use is strictly **prohibited** for conceptual exercises.
   - For applied (Python) exercises, you may only use AI for **debugging**. Do not use AI to generate answers.
- **Due Date: November 13, 2026, by 11:59 PM.**

# Exercise 1 (4 points)

A facilitator wants an elicitation method that automatically forces each stakeholder's probabilities to sum to 1 and discourages degenerate 0/1 and 50/50 answers, while staying quick enough for a small decision advisory committee. Which method best fits these requirements?

- (a) Conjoint analysis (experimental design-based)
- (b) Discrete choice analysis (experimental design-based)
- (c) Direct questioning with a Likert scale
- (d) Chip assignment (activity-based)
- (e) None of the above

D) 

# Exercise 2 (5 points)

Two decision advisory committees each produce the same consensus (mean) prior of $(0.15, 0.30, 0.55)$ over the outcomes No Development, Modest, and Major. In the first committee every member reported almost exactly $(0.15, 0.30, 0.55)$. In the second committee, half of the members reported $(0, 0, 1)$ and the other half reported $(0.30, 0.60, 0.10)$. Explain why reporting only the mean can be misleading here, and describe what else you would report (and why) before recommending a decision based on these priors.

Reporting only the mean can be misleading because two committees can have the same average probabilities but very different levels of agreement among their members.

In the first committee, members reported nearly identical probabilities of (0.15, 0.30, 0.55), showing strong agreement and relatively low disagreement.

In the second committee, half the members reported (0, 0, 1), while the other half reported (0.30, 0.60, 0.10). Although the average is the same, the members have very different opinions and levels of confidence.

# Exercise 3 (12 points)

A decision advisory committee of 12 stakeholders each distributed 20 chips among three hotel development options — No Development, Modest, and Major — to express their prior beliefs. Run the snippet below to generate the normalized chip allocations (use these exact values; do not change the seed):

```
import numpy as np
import pandas as pd

rng = np.random.default_rng(511)
n_members = 12
options = ['NoDev', 'Modest', 'Major']
allocs = rng.dirichlet([1.5, 3.0, 5.5], size=n_members)
prob_df = pd.DataFrame(allocs, columns=options)
```

- (a) **Consensus (4 points):** Compute the equal-weight consensus (mean) prior across the 12 stakeholders for each option, and report which option the committee believes is most likely.
- (b) **Bootstrap confidence intervals (8 points):** Resample the stakeholders (rows) with replacement $B = 5000$ times using `np.random.default_rng(511)`. For each resample, compute the mean prior for each option, then form a 95% percentile confidence interval (2.5th and 97.5th percentiles) for the consensus probability of each option. Report the mean together with its lower and upper bounds in a single table.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(511)
n_members = 12

options = ['NoDev', 'Modest', 'Major']

allocs = rng.dirichlet([1.5, 3.0, 5.5], size=n_members)

prob_df = pd.DataFrame(allocs, columns=options)

print(prob_df)

       NoDev    Modest     Major
0   0.102502  0.453420  0.444078
1   0.245202  0.247434  0.507364
2   0.084673  0.302672  0.612655
3   0.025133  0.320630  0.654237
4   0.030492  0.079627  0.889881
5   0.005117  0.392857  0.602026
6   0.157917  0.169359  0.672724
7   0.005222  0.509550  0.485228
8   0.133057  0.305869  0.561073
9   0.017478  0.390451  0.592071
10  0.171763  0.260630  0.567606
11  0.026957  0.335584  0.637459


In [2]:
consensus = prob_df.mean()

print("Consensus Prior:")
print(consensus)

print("\nMost Likely Development Option:")
print(consensus.idxmax())

Consensus Prior:
NoDev     0.083793
Modest    0.314007
Major     0.602200
dtype: float64

Most Likely Development Option:
Major


In [3]:
B = 5000
rng_boot = np.random.default_rng(511)

boot_means = np.empty((B, len(options)))

for b in range(B):
    sample = prob_df.iloc[
        rng_boot.integers(0, n_members, size=n_members)
    ]
    boot_means[b] = sample.mean().values

lower = np.percentile(boot_means, 2.5, axis=0)
upper = np.percentile(boot_means, 97.5, axis=0)

results = pd.DataFrame({
    "Mean": consensus.values,
    "Lower 95% CI": lower,
    "Upper 95% CI": upper
}, index=options)

print(results.round(4))

          Mean  Lower 95% CI  Upper 95% CI
NoDev   0.0838        0.0437        0.1301
Modest  0.3140        0.2493        0.3759
Major   0.6022        0.5470        0.6687


# Exercise 4 (4 points)

An analyst replaces every uncertain input in a decision model by its average (mean) value and then optimizes the resulting deterministic model. For a nonlinear outcome, this "optimize-at-the-mean" approach generally produces a biased estimate of the true expected outcome. Which result formalizes why this bias occurs?

- (a) The Central Limit Theorem
- (b) Jensen's inequality
- (c) Bayes' theorem
- (d) The Law of Large Numbers
- (e) None of the above

B) 

# Exercise 5 (5 points)

For a one-shot, high-stakes decision, a manager proposes choosing the option with the smallest Value at Risk (VaR) at the 95% level. Define both Value at Risk (VaR$_\alpha$) and Conditional Value at Risk (CVaR$_\alpha$) for a loss distribution, explain why optimizing VaR alone can be dangerous for such a decision, and state why CVaR is generally the preferred risk measure.

Value at Risk (VaR) at the 95% level is the maximum loss threshold that is not exceeded in 95% of possible outcomes. It represents the 95th percentile of the loss distribution.

Conditional Value at Risk (CVaR) is the average loss among the worst 5% of outcomes beyond the VaR threshold.

Optimizing VaR alone can be dangerous because it does not measure how severe losses could become after the VaR threshold is exceeded. Two decisions could have the same VaR but very different potential catastrophic losses.

CVaR is generally preferred because it considers the average severity of losses in the worst 5% of outcomes. This provides a more complete picture of risk and helps managers make better decisions when dealing with high-stakes situations.

# Exercise 6 (12 points)

A retailer must order $q$ units of a perishable product **before** observing demand $D$, where demand is random with $D \sim \text{Normal}(\mu = 100,\ \sigma = 25)$. Each unit sells for $r = \$10$, costs $c = \$4$ to buy, and leftover units salvage for $s = \$2$. The underage and overage costs are $c_u = r - c$ and $c_o = c - s$, and realized profit is

$$\pi(q, D) = r\,\min(q, D) + s\,(q - D)^{+} - c\,q.$$

- (a) **Critical fractile (4 points):** Compute $c_u$, $c_o$, and the critical ratio $c_u / (c_u + c_o)$. Then use `scipy.stats.norm.ppf` to compute the critical-fractile optimal order quantity $q^{*} = F^{-1}\!\big(c_u / (c_u + c_o)\big)$ and report it.
- (b) **Verify by simulation (8 points):** Draw 200,000 demand scenarios using `np.random.default_rng(2026)` (clip values at 0). Compute the expected profit for each order quantity on the grid `np.arange(70, 141, 1.0)`, find the simulation-optimal order quantity, and confirm it matches $q^{*}$ from part (a). Does $q^{*}$ equal the mean demand of 100? Briefly explain why or why not.

In [4]:
import numpy as np
from scipy.stats import norm

r = 10
c = 4
s = 2
mu = 100
sigma = 25

cu = r - c
co = c - s

critical_ratio = cu / (cu + co)

q_star = norm.ppf(critical_ratio, loc=mu, scale=sigma)

print(f"Underage Cost: ${cu:.2f}")
print(f"Overage Cost: ${co:.2f}")
print(f"Critical Ratio: {critical_ratio:.2f}")
print(f"Optimal Order Quantity: {q_star:.2f}")

Underage Cost: $6.00
Overage Cost: $2.00
Critical Ratio: 0.75
Optimal Order Quantity: 116.86


In [5]:
rng = np.random.default_rng(2026)

demand = rng.normal(100, 25, 200_000)
demand = np.clip(demand, 0, None)

q_grid = np.arange(70, 141, 1.0)

expected_profits = []

for q in q_grid:
    profit = (
        r * np.minimum(q, demand)
        + s * np.maximum(q - demand, 0)
        - c * q
    )

    expected_profits.append(profit.mean())

best_index = np.argmax(expected_profits)

best_q = q_grid[best_index]
best_profit = expected_profits[best_index]

print(f"Simulation Optimal Order Quantity: {best_q:.0f}")
print(f"Maximum Expected Profit: ${best_profit:.2f}")
print(f"Critical Fractile Order Quantity: {q_star:.2f}")

Simulation Optimal Order Quantity: 117
Maximum Expected Profit: $536.75
Critical Fractile Order Quantity: 116.86
